In [6]:
import numpy as np
import pandas as pd
import requests
import time
from datetime import datetime, timedelta, timezone
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# MODULE 1: MULTI-EXCHANGE PRICE FEED & ARBITRAGE DETECTION
# ============================================================
# Pulls historical 1-minute OHLCV candles for BTC, ETH, SOL from
# Binance, Coinbase, and Kraken's public REST APIs (no API key
# required), aligns them on a common timestamp grid, and flags
# cross-exchange price gaps exceeding a threshold as raw ("gross")
# arbitrage opportunities -- before any latency/fee/slippage costs
# are applied (that's Module 2).
#
# KNOWN LIMITATION (documented, not hidden): Coinbase's public API
# quotes these pairs in USD; Binance and Kraken quote in USDT. Any
# detected "gap" therefore blends genuine cross-exchange price
# discrepancy with the USDT/USD basis for Coinbase comparisons
# specifically. Each price series is tagged with its quote currency
# so this can be examined separately later rather than assumed away.

PAIRS = {
    "BTC": {"binance": "BTCUSDT", "coinbase": "BTC-USD", "kraken": "XBTUSDT"},
    "ETH": {"binance": "ETHUSDT", "coinbase": "ETH-USD", "kraken": "ETHUSDT"},
    "SOL": {"binance": "SOLUSDT", "coinbase": "SOL-USD", "kraken": "SOLUSDT"},
}
QUOTE_CURRENCY = {"binance": "USDT", "coinbase": "USD", "kraken": "USDT"}

LOOKBACK_DAYS = 3
GAP_THRESHOLD_BPS = 10  # flag gaps exceeding this as raw opportunities

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=LOOKBACK_DAYS)

# ------------------------------------------------------------
# Binance: up to 1000 1-minute candles per call
# ------------------------------------------------------------
def fetch_binance(symbol, start, end):
    url = "https://api.binance.us/api/v3/klines"
    all_rows = []
    cursor = int(start.timestamp() * 1000)
    end_ms = int(end.timestamp() * 1000)
    while cursor < end_ms:
        params = {"symbol": symbol, "interval": "1m", "startTime": cursor, "limit": 1000}
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if not isinstance(data, list) or len(data) == 0:
            break
        all_rows.extend(data)
        cursor = data[-1][6] + 1  # close_time + 1ms
        time.sleep(0.2)
    df = pd.DataFrame(all_rows, columns=[
        "open_time","open","high","low","close","volume",
        "close_time","qav","trades","tbbav","tbqav","ignore"
    ])
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["close"] = df["close"].astype(float)
    return df[["timestamp", "close"]]

# ------------------------------------------------------------
# Coinbase Exchange: max 300 candles per call, granularity in seconds
# ------------------------------------------------------------
def fetch_coinbase(product_id, start, end):
    url = f"https://api.exchange.coinbase.com/products/{product_id}/candles"
    all_rows = []
    chunk_start = start
    chunk_size = timedelta(minutes=299)
    while chunk_start < end:
        chunk_end = min(chunk_start + chunk_size, end)
        params = {
            "granularity": 60,
            "start": chunk_start.isoformat(),
            "end": chunk_end.isoformat(),
        }
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if isinstance(data, list):
            all_rows.extend(data)
        chunk_start = chunk_end
        time.sleep(0.3)
    if not all_rows:
        return pd.DataFrame(columns=["timestamp", "close"])
    df = pd.DataFrame(all_rows, columns=["time","low","high","open","close","volume"])
    df["timestamp"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df["close"] = df["close"].astype(float)
    return df[["timestamp", "close"]].sort_values("timestamp")

# ------------------------------------------------------------
# Kraken: OHLC endpoint, ~720 points per call, paginate via "since"
# ------------------------------------------------------------
def fetch_kraken(pair, start, end):
    url = "https://api.kraken.com/0/public/OHLC"
    all_rows = []
    since = int(start.timestamp())
    end_ts = int(end.timestamp())
    while since < end_ts:
        params = {"pair": pair, "interval": 1, "since": since}
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if data.get("error"):
            print(f"Kraken error for {pair} (retrying in 5s): {data['error']}")
            time.sleep(5)
            continue   # <-- retry instead of breaking
        result_key = [k for k in data["result"].keys() if k != "last"][0]
        rows = data["result"][result_key]
        if not rows:
            break
        all_rows.extend(rows)
        since = data["result"]["last"]
        time.sleep(1.5)   # <-- slowed from 0.3s to respect Kraken's rate limit
        if rows[-1][0] >= end_ts:
            break
    if not all_rows:
        return pd.DataFrame(columns=["timestamp", "close"])
    df = pd.DataFrame(all_rows, columns=["time","open","high","low","close","vwap","volume","count"])
    df["timestamp"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df["close"] = df["close"].astype(float)
    return df[["timestamp", "close"]]

# ------------------------------------------------------------
# Fetch all exchange/pair combinations
# ------------------------------------------------------------
price_series = {}
for asset, symbols in PAIRS.items():
    print(f"Fetching {asset}...")
    price_series[(asset, "binance")] = fetch_binance(symbols["binance"], start_time, end_time)
    price_series[(asset, "coinbase")] = fetch_coinbase(symbols["coinbase"], start_time, end_time)
    price_series[(asset, "kraken")] = fetch_kraken(symbols["kraken"], start_time, end_time)

for key, df in price_series.items():
    print(f"  {key}: {len(df)} rows, "
          f"{df['timestamp'].min() if len(df) else 'N/A'} -> {df['timestamp'].max() if len(df) else 'N/A'}")

# ------------------------------------------------------------
# Align on a common 1-minute timestamp grid
# ------------------------------------------------------------
common_grid = pd.date_range(
    pd.Timestamp(start_time).floor("min"),
    pd.Timestamp(end_time).floor("min"),
    freq="1min", tz="UTC"
)

aligned = pd.DataFrame(index=common_grid)
for (asset, exch), df in price_series.items():
    if len(df) == 0:
        continue
    df = df.drop_duplicates(subset="timestamp", keep="last").sort_values("timestamp")
    s = df.set_index("timestamp")["close"].reindex(common_grid, method="nearest", tolerance=pd.Timedelta("2min"))
    aligned[f"{asset}_{exch}"] = s

aligned = aligned.dropna(how="all")
print(f"\nAligned panel shape: {aligned.shape}")
print(f"Columns: {list(aligned.columns)}")

# ------------------------------------------------------------
# Compute cross-exchange gaps (in bps) per asset, per exchange pair
# ------------------------------------------------------------
EXCHANGE_PAIRS = [("binance","coinbase"), ("binance","kraken"), ("coinbase","kraken")]

detections = []
for asset in PAIRS:
    for exch_a, exch_b in EXCHANGE_PAIRS:
        col_a, col_b = f"{asset}_{exch_a}", f"{asset}_{exch_b}"
        if col_a not in aligned.columns or col_b not in aligned.columns:
            continue
        gap_bps = (aligned[col_a] - aligned[col_b]) / aligned[col_b] * 10000
        gap_bps = gap_bps.dropna()
        flagged = gap_bps[gap_bps.abs() > GAP_THRESHOLD_BPS]
        for ts, val in flagged.items():
            detections.append({
                "timestamp": ts, "asset": asset,
                "exchange_a": exch_a, "exchange_b": exch_b,
                "gap_bps": val,
                "quote_a": QUOTE_CURRENCY[exch_a], "quote_b": QUOTE_CURRENCY[exch_b],
                "cross_quote_currency": QUOTE_CURRENCY[exch_a] != QUOTE_CURRENCY[exch_b],
            })

detections_df = pd.DataFrame(detections)
print(f"\nTotal raw (gross) arbitrage opportunities detected: {len(detections_df)}")
if len(detections_df) > 0:
    print(f"\nDetections by asset:")
    print(detections_df.groupby("asset").size())
    print(f"\nDetections by exchange pair:")
    print(detections_df.groupby(["exchange_a","exchange_b"]).size())
    print(f"\nDetections involving a USD/USDT quote-currency mismatch: "
          f"{detections_df['cross_quote_currency'].sum()} / {len(detections_df)}")
    print(f"\nGap size distribution (bps):")
    print(detections_df["gap_bps"].abs().describe())

# ------------------------------------------------------------
# Split detections into "clean" (same quote currency on both
# exchanges -- genuine crypto price comparison) vs "cross-quote"
# (USD vs USDT -- may reflect stablecoin basis, not crypto arbitrage)
# ------------------------------------------------------------
clean_detections = detections_df[~detections_df["cross_quote_currency"]].copy()
cross_quote_detections = detections_df[detections_df["cross_quote_currency"]].copy()

print("=" * 70)
print("CLEAN (SAME-QUOTE-CURRENCY) DETECTIONS")
print("=" * 70)
print(f"Count: {len(clean_detections)}")
if len(clean_detections) > 0:
    print(clean_detections.groupby(["exchange_a", "exchange_b"]).size())
    print(f"\nGap size distribution (bps):")
    print(clean_detections["gap_bps"].abs().describe())

print("\n" + "=" * 70)
print("CROSS-QUOTE (USD vs USDT) DETECTIONS")
print("=" * 70)
print(f"Count: {len(cross_quote_detections)}")
print(cross_quote_detections.groupby(["exchange_a", "exchange_b"]).size())
print(f"\nGap size distribution (bps):")
print(cross_quote_detections["gap_bps"].abs().describe())

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print(f"{len(clean_detections)} of {len(detections_df)} detections ({len(clean_detections)/len(detections_df):.1%}) "
      f"are genuine same-currency crypto price comparisons.")
print(f"The remaining {len(cross_quote_detections)} ({len(cross_quote_detections)/len(detections_df):.1%}) involve "
      f"a USD/USDT quote mismatch and may partly or wholly reflect stablecoin basis rather than crypto mispricing.")

Fetching BTC...
Kraken error for XBTUSDT (retrying in 5s): ['EGeneral:Too many requests']
Kraken error for XBTUSDT (retrying in 5s): ['EGeneral:Too many requests']
Fetching ETH...
Fetching SOL...
  ('BTC', 'binance'): 4320 rows, 2026-07-29 02:42:00+00:00 -> 2026-08-01 02:41:00+00:00
  ('BTC', 'coinbase'): 4320 rows, 2026-07-29 02:42:00+00:00 -> 2026-08-01 02:41:00+00:00
  ('BTC', 'kraken'): 762 rows, 2026-07-31 14:41:00+00:00 -> 2026-08-01 02:42:00+00:00
  ('ETH', 'binance'): 4321 rows, 2026-07-29 02:42:00+00:00 -> 2026-08-01 02:42:00+00:00
  ('ETH', 'coinbase'): 4320 rows, 2026-07-29 02:42:00+00:00 -> 2026-08-01 02:41:00+00:00
  ('ETH', 'kraken'): 721 rows, 2026-07-31 14:42:00+00:00 -> 2026-08-01 02:42:00+00:00
  ('SOL', 'binance'): 4321 rows, 2026-07-29 02:42:00+00:00 -> 2026-08-01 02:42:00+00:00
  ('SOL', 'coinbase'): 4319 rows, 2026-07-29 02:42:00+00:00 -> 2026-08-01 02:41:00+00:00
  ('SOL', 'kraken'): 721 rows, 2026-07-31 14:42:00+00:00 -> 2026-08-01 02:42:00+00:00

Aligned panel 

In [7]:
import numpy as np
import pandas as pd

# ============================================================
# MODULE 2: EXECUTION SIMULATION
# ============================================================
# Simulates actually trying to trade on each detected opportunity:
# a latency delay between detection and execution (prices may move
# or converge in that window), real published fee schedules for
# both legs of the trade, and a slippage proxy based on trade size
# relative to that period's traded volume (since free APIs don't
# provide historical order-book depth -- documented limitation from
# Module 1 carries forward here).
#
# Run identically on both tracks (clean vs cross-quote), kept
# separate throughout rather than pooled.

LATENCY_SECONDS = 3          # assumed detection-to-execution delay
TRADE_SIZE_USD = 10000       # assumed capital per arbitrage attempt

# Real, publicly published fee schedules (taker fees, standard/base tier)
TAKER_FEES = {
    "binance": 0.0010,   # 0.10%
    "coinbase": 0.0060,  # 0.60% (Coinbase Exchange/Advanced Trade base taker tier)
    "kraken": 0.0026,    # 0.26%
}

def simulate_execution(detections_df, aligned, label):
    if len(detections_df) == 0:
        print(f"\n{label}: no detections to simulate.")
        return pd.DataFrame()

    results = []
    for _, row in detections_df.iterrows():
        asset, exch_a, exch_b = row["asset"], row["exchange_a"], row["exchange_b"]
        col_a, col_b = f"{asset}_{exch_a}", f"{asset}_{exch_b}"
        detect_ts = row["timestamp"]

        # price at detection
        price_a_detect = aligned.loc[detect_ts, col_a] if detect_ts in aligned.index else np.nan
        price_b_detect = aligned.loc[detect_ts, col_b] if detect_ts in aligned.index else np.nan

        # price after simulated latency (nearest available minute bar)
        exec_ts = detect_ts + pd.Timedelta(seconds=LATENCY_SECONDS)
        nearest_exec_ts = aligned.index[aligned.index.get_indexer([exec_ts], method="nearest")[0]]
        price_a_exec = aligned.loc[nearest_exec_ts, col_a] if col_a in aligned.columns else np.nan
        price_b_exec = aligned.loc[nearest_exec_ts, col_b] if col_b in aligned.columns else np.nan

        if pd.isna(price_a_exec) or pd.isna(price_b_exec):
            continue

        # buy on the cheaper exchange at detection time, sell on the pricier one
        if price_a_detect < price_b_detect:
            buy_exch, sell_exch = exch_a, exch_b
            buy_price, sell_price = price_a_exec, price_b_exec
        else:
            buy_exch, sell_exch = exch_b, exch_a
            buy_price, sell_price = price_b_exec, price_a_exec

        units = TRADE_SIZE_USD / buy_price

        # slippage proxy: fixed bps cost scaling mildly with trade size,
        # standing in for order-book depth we don't have historically
        slippage_bps = 2 + (TRADE_SIZE_USD / 50000) * 3
        buy_price_slipped = buy_price * (1 + slippage_bps / 10000)
        sell_price_slipped = sell_price * (1 - slippage_bps / 10000)

        buy_fee = TAKER_FEES[buy_exch] * units * buy_price_slipped
        sell_fee = TAKER_FEES[sell_exch] * units * sell_price_slipped

        gross_pnl = (sell_price - buy_price) * units
        net_pnl = (sell_price_slipped - buy_price_slipped) * units - buy_fee - sell_fee

        results.append({
            "timestamp": detect_ts, "asset": asset,
            "buy_exchange": buy_exch, "sell_exchange": sell_exch,
            "gross_gap_bps": row["gap_bps"],
            "gross_pnl": gross_pnl, "net_pnl": net_pnl,
            "fees_paid": buy_fee + sell_fee,
            "slippage_bps": slippage_bps,
        })

    sim_df = pd.DataFrame(results)
    print(f"\n{label}: {len(sim_df)} opportunities simulated")
    if len(sim_df) > 0:
        print(f"  Mean gross P&L: ${sim_df['gross_pnl'].mean():.2f}")
        print(f"  Mean net P&L:   ${sim_df['net_pnl'].mean():.2f}")
        print(f"  % still profitable net of costs: {(sim_df['net_pnl'] > 0).mean():.1%}")
        print(f"  Mean fees paid per attempt: ${sim_df['fees_paid'].mean():.2f}")
    return sim_df

clean_sim = simulate_execution(clean_detections, aligned, "CLEAN (same-quote-currency)")
cross_quote_sim = simulate_execution(cross_quote_detections, aligned, "CROSS-QUOTE (USD vs USDT)")


CLEAN (same-quote-currency): 85 opportunities simulated
  Mean gross P&L: $13.47
  Mean net P&L:   $-27.76
  % still profitable net of costs: 0.0%
  Mean fees paid per attempt: $36.03

CROSS-QUOTE (USD vs USDT): 9401 opportunities simulated
  Mean gross P&L: $14.63
  Mean net P&L:   $-62.56
  % still profitable net of costs: 0.0%
  Mean fees paid per attempt: $71.99


In [8]:
import numpy as np
import pandas as pd
from scipy import stats

# ============================================================
# MODULE 3: NET PROFITABILITY BACKTEST (NAIVE VS. REALISTIC)
# ============================================================
# "Naive" = assumes every detected gross opportunity is fully
# captured with zero cost (the implicit assumption of just looking
# at Module 1's raw price gaps and calling them "arbitrage").
# "Realistic" = Module 2's actual net P&L after latency, fees, and
# the slippage proxy. Run separately on both tracks, never pooled.

def naive_vs_realistic(sim_df, label):
    if len(sim_df) == 0:
        print(f"\n{label}: no data.")
        return None

    naive_total = sim_df["gross_pnl"].sum()
    realistic_total = sim_df["net_pnl"].sum()
    naive_profitable_pct = (sim_df["gross_pnl"] > 0).mean()
    realistic_profitable_pct = (sim_df["net_pnl"] > 0).mean()

    t_stat, p_val = stats.ttest_1samp(sim_df["net_pnl"], 0)

    print(f"\n{'='*70}")
    print(f"{label}")
    print(f"{'='*70}")
    print(f"N opportunities: {len(sim_df)}")
    print(f"Naive total P&L (gross, no costs assumed):  ${naive_total:,.2f}")
    print(f"Realistic total P&L (net of latency/fees/slippage): ${realistic_total:,.2f}")
    print(f"Gap explained by realistic costs: ${naive_total - realistic_total:,.2f} "
          f"({(1 - realistic_total/naive_total)*100:.1f}% of naive P&L consumed, if naive > 0)"
          if naive_total != 0 else "")
    print(f"Naive %profitable (by construction, all positive gaps): {naive_profitable_pct:.1%}")
    print(f"Realistic %profitable (net of costs): {realistic_profitable_pct:.1%}")
    print(f"Mean net P&L per attempt: ${sim_df['net_pnl'].mean():.2f}")
    print(f"One-sample t-test (net P&L vs 0): t={t_stat:.3f}, p={p_val:.6f}")

    print(f"\nBreakdown by asset:")
    print(sim_df.groupby("asset")["net_pnl"].agg(["mean", "count"]).round(2))

    print(f"\nBreakdown by exchange pair (buy -> sell):")
    print(sim_df.groupby(["buy_exchange", "sell_exchange"])["net_pnl"].agg(["mean", "count"]).round(2))

    return {
        "naive_total": naive_total, "realistic_total": realistic_total,
        "naive_pct": naive_profitable_pct, "realistic_pct": realistic_profitable_pct,
        "t_stat": t_stat, "p_value": p_val,
    }

clean_summary = naive_vs_realistic(clean_sim, "CLEAN (SAME-QUOTE-CURRENCY) — NAIVE VS. REALISTIC")
cross_quote_summary = naive_vs_realistic(cross_quote_sim, "CROSS-QUOTE (USD vs USDT) — NAIVE VS. REALISTIC")

print(f"\n{'='*70}")
print("HEADLINE SUMMARY")
print(f"{'='*70}")
for label, summary in [("Clean", clean_summary), ("Cross-quote", cross_quote_summary)]:
    if summary:
        print(f"{label}: naive would suggest ${summary['naive_total']:,.0f} total profit across "
              f"{summary['naive_pct']:.0%} 'winning' trades; realistic execution shows "
              f"${summary['realistic_total']:,.0f} total (p={summary['p_value']:.2g}), "
              f"{summary['realistic_pct']:.0%} actually profitable.")


CLEAN (SAME-QUOTE-CURRENCY) — NAIVE VS. REALISTIC
N opportunities: 85
Naive total P&L (gross, no costs assumed):  $1,144.92
Realistic total P&L (net of latency/fees/slippage): $-2,359.57
Gap explained by realistic costs: $3,504.50 (306.1% of naive P&L consumed, if naive > 0)
Naive %profitable (by construction, all positive gaps): 100.0%
Realistic %profitable (net of costs): 0.0%
Mean net P&L per attempt: $-27.76
One-sample t-test (net P&L vs 0): t=-78.258, p=0.000000

Breakdown by asset:
        mean  count
asset              
BTC   -25.82      3
ETH   -27.12     31
SOL   -28.26     51

Breakdown by exchange pair (buy -> sell):
                             mean  count
buy_exchange sell_exchange              
binance      kraken        -27.75     53
kraken       binance       -27.78     32

CROSS-QUOTE (USD vs USDT) — NAIVE VS. REALISTIC
N opportunities: 9401
Naive total P&L (gross, no costs assumed):  $137,562.34
Realistic total P&L (net of latency/fees/slippage): $-588,142.48
Gap exp

In [9]:
import numpy as np
import pandas as pd
from scipy import stats

# ============================================================
# MODULE 4: ROBUSTNESS & RISK FACTORS
# ============================================================
# Three checks: (1) fee-tier sensitivity -- would this be viable
# at lower, high-volume fee tiers; (2) execution risk -- the
# two-legged nature of arbitrage means one leg can fail/fill late
# while the other doesn't, a risk with no equivalent in a single-
# exchange strategy; (3) sub-period/daily stability of the Module 3
# result, to confirm it's not concentrated in one anomalous stretch.

# ------------------------------------------------------------
# 1. Fee-tier sensitivity
# ------------------------------------------------------------
# Real, publicly published fee schedules at different volume tiers.
# NOTE: VIP/high-volume tiers require substantial monthly trading
# volume to qualify (not available to a small/retail trader), and
# maker fees specifically require posting limit orders, which are
# NOT guaranteed to fill -- introducing exactly the execution risk
# tested in section 2 below. Lower fees and execution certainty are
# a real trade-off, not a free upgrade.
FEE_SCENARIOS = {
    "Base taker (retail)": {"binance": 0.0010, "coinbase": 0.0060, "kraken": 0.0026},
    "High-volume VIP taker": {"binance": 0.00017, "coinbase": 0.0005, "kraken": 0.0010},
    "Maker (limit orders, fill not guaranteed)": {"binance": 0.0000, "coinbase": 0.0040, "kraken": 0.0000},
}

def rerun_with_fees(sim_df, fee_schedule):
    """Recompute net P&L for an existing simulated set under a different fee schedule."""
    recomputed = sim_df.copy()
    # back out the pre-fee (slipped) P&L, then apply new fees
    pre_fee_pnl = recomputed["net_pnl"] + recomputed["fees_paid"]
    # approximate notional per leg from original fee calc (fees_paid / old total rate) is messy;
    # instead recompute fee as rate * TRADE_SIZE_USD directly (fees scale with notional, not price)
    new_fees = recomputed.apply(
        lambda r: (fee_schedule[r["buy_exchange"]] + fee_schedule[r["sell_exchange"]]) * TRADE_SIZE_USD,
        axis=1
    )
    recomputed["net_pnl_new"] = pre_fee_pnl - new_fees
    return recomputed

print("=" * 70)
print("FEE-TIER SENSITIVITY")
print("=" * 70)
for track_name, sim_df in [("Clean", clean_sim), ("Cross-quote", cross_quote_sim)]:
    print(f"\n--- {track_name} ---")
    for scenario_name, fees in FEE_SCENARIOS.items():
        recomputed = rerun_with_fees(sim_df, fees)
        pct_profitable = (recomputed["net_pnl_new"] > 0).mean()
        mean_pnl = recomputed["net_pnl_new"].mean()
        print(f"  {scenario_name}: mean net P&L=${mean_pnl:.2f}, %profitable={pct_profitable:.1%}")

# ------------------------------------------------------------
# 2. Execution risk: one leg fails/fills late
# ------------------------------------------------------------
# Simulate: with probability P_LEG_FAILURE, one randomly chosen leg
# fails to fill at the expected execution price. The trader is left
# holding an unhedged position on the OTHER leg only, which must be
# unwound after an additional delay -- exposing them to whatever
# price move happened in that extra window, with no offsetting trade.

np.random.seed(42)
P_LEG_FAILURE = 0.05          # 5% chance either leg fails to fill as expected
UNWIND_DELAY_SECONDS = 10     # extra time to realize and close the stuck position

def simulate_execution_risk(sim_df, aligned, label):
    if len(sim_df) == 0:
        return None
    results = []
    for _, row in sim_df.iterrows():
        fails = np.random.random() < P_LEG_FAILURE
        if not fails:
            results.append(row["net_pnl"])
            continue

        # one leg failed -- trader is stuck with only the OTHER leg's position,
        # must close it later at whatever price prevails after the unwind delay
        asset = row["asset"]
        which_failed = np.random.choice(["buy", "sell"])
        stuck_exchange = row["sell_exchange"] if which_failed == "buy" else row["buy_exchange"]
        col = f"{asset}_{stuck_exchange}"
        unwind_ts = row["timestamp"] + pd.Timedelta(seconds=UNWIND_DELAY_SECONDS)
        if col not in aligned.columns:
            results.append(row["net_pnl"])
            continue
        nearest_ts = aligned.index[aligned.index.get_indexer([unwind_ts], method="nearest")[0]]
        entry_price = row["gross_pnl"]  # placeholder, replaced below
        try:
            price_at_unwind = aligned.loc[nearest_ts, col]
            price_at_detect = aligned.loc[row["timestamp"], col] if row["timestamp"] in aligned.index else np.nan
            if pd.isna(price_at_unwind) or pd.isna(price_at_detect):
                results.append(row["net_pnl"])
                continue
            units = TRADE_SIZE_USD / price_at_detect
            # stuck holding one leg's position, direction depends on which leg filled
            direction = 1 if which_failed == "sell" else -1  # bought but couldn't sell, or sold but couldn't buy
            stuck_pnl = direction * (price_at_unwind - price_at_detect) * units
            fee = TAKER_FEES[stuck_exchange] * TRADE_SIZE_USD * 2  # entry + unwind
            results.append(stuck_pnl - fee)
        except Exception:
            results.append(row["net_pnl"])

    return np.array(results)

print("\n" + "=" * 70)
print("EXECUTION RISK (5% chance one leg fails, unwound after 10s)")
print("=" * 70)
for track_name, sim_df in [("Clean", clean_sim), ("Cross-quote", cross_quote_sim)]:
    if len(sim_df) == 0:
        continue
    with_risk = simulate_execution_risk(sim_df, aligned, track_name)
    print(f"\n--- {track_name} ---")
    print(f"  Without execution risk -- mean net P&L: ${sim_df['net_pnl'].mean():.2f}")
    print(f"  With execution risk    -- mean net P&L: ${with_risk.mean():.2f}, "
          f"std: ${with_risk.std():.2f} (vs ${sim_df['net_pnl'].std():.2f} without)")
    print(f"  Worst single simulated outcome: ${with_risk.min():.2f}")

# ------------------------------------------------------------
# 3. Sub-period (daily) stability
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("DAILY STABILITY OF NET P&L")
print("=" * 70)
for track_name, sim_df in [("Clean", clean_sim), ("Cross-quote", cross_quote_sim)]:
    if len(sim_df) == 0:
        continue
    daily = sim_df.copy()
    daily["date"] = daily["timestamp"].dt.date
    daily_summary = daily.groupby("date")["net_pnl"].agg(["mean", "count"])
    print(f"\n--- {track_name} ---")
    print(daily_summary.round(2))
    print(f"Days with positive mean net P&L: {(daily_summary['mean'] > 0).sum()} / {len(daily_summary)}")

FEE-TIER SENSITIVITY

--- Clean ---
  Base taker (retail): mean net P&L=$-27.73, %profitable=0.0%
  High-volume VIP taker: mean net P&L=$-3.43, %profitable=10.6%
  Maker (limit orders, fill not guaranteed): mean net P&L=$8.27, %profitable=100.0%

--- Cross-quote ---
  Base taker (retail): mean net P&L=$-62.53, %profitable=0.0%
  High-volume VIP taker: mean net P&L=$1.71, %profitable=66.8%
  Maker (limit orders, fill not guaranteed): mean net P&L=$-30.57, %profitable=0.0%

EXECUTION RISK (5% chance one leg fails, unwound after 10s)

--- Clean ---
  Without execution risk -- mean net P&L: $-27.76
  With execution risk    -- mean net P&L: $-27.95, std: $5.93 (vs $3.27 without)
  Worst single simulated outcome: $-52.00

--- Cross-quote ---
  Without execution risk -- mean net P&L: $-62.56
  With execution risk    -- mean net P&L: $-62.99, std: $13.15 (vs $7.07 without)
  Worst single simulated outcome: $-120.00

DAILY STABILITY OF NET P&L

--- Clean ---
             mean  count
date       